In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import binom
import matplotlib.pyplot as plt

# KONFIGURACJA

DESKTOP_PATH = os.path.join(os.path.expanduser("~"), "Desktop")
SCIEZKA_CR = os.path.join(DESKTOP_PATH, "oulu_1H_data.csv")

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
FOLDER_DANE = os.path.join(BASE_DIR, "..", "dane")
SCIEZKA_EQ = os.path.join(FOLDER_DANE, "eq_data_orginal_light.csv")

WYJSCIE_WYKRES = os.path.join(BASE_DIR, f"slajd10_extreme_ppdf_{STACJA}.png")
WYJSCIE_CSV = os.path.join(BASE_DIR, f"slajd10_extreme_ppdf_{STACJA}.csv")

BIN_SIZE_MINUTES = 1440          # 1 dzień
WINDOW_DAYS = 3350               # ~9 lat
WINDOWS_COUNT = round(WINDOW_DAYS * 1440 / BIN_SIZE_MINUTES)

LAG_NOT_OPTIMIZED_MIN = 0
LAG_OPTIMIZED_MIN = 28600        # wartość ze slajdu 10

SHIFT_NOT_OPT_BINS = round(LAG_NOT_OPTIMIZED_MIN / BIN_SIZE_MINUTES)
SHIFT_OPT_BINS = round(LAG_OPTIMIZED_MIN / BIN_SIZE_MINUTES)

MIN_N = 100                      # minimalna liczba obserwacji


# DANE CR
def wczytaj_dane_cr_1h(sciezka):

    df = pd.read_csv(sciezka)

    # Konwersja na datetime i ustawienie jako indeks
    df["date"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
    if df["date"].dt.tz is not None:
        df["date"] = df["date"].dt.tz_localize(None)
    df = df.dropna(subset=["date"])
    df = df.set_index("date").sort_index()

    # Origin = dokładny początek pliku CR
    origin = df.index.min()

    # Resampling do 1-dniowych binów
    seria = df["value"].resample(f"{BIN_SIZE_MINUTES}min", origin=origin).mean()

    # Interpolacja tylko krótkich luk (max 5 dni)
    MAX_GAP_BINS = 5
    seria = seria.interpolate(method="linear", limit=MAX_GAP_BINS, limit_direction="both")

    return seria, origin


# DANE EQ
def wczytaj_dane_eq(sciezka, origin):

    df = pd.read_csv(sciezka)

    # Konwersja na datetime i ustawienie jako indeks
    df["date"] = pd.to_datetime(df["time"], errors="coerce", utc=True)
    if df["date"].dt.tz is not None:
        df["date"] = df["date"].dt.tz_localize(None)
    df = df.dropna(subset=["date"])

    # Filtracja trzęsień o magnitudzie >= 4.0
    eq_m4 = df[df["mag"] >= 4.0].set_index("date").sort_index()

    # Resampling do 1-dniowych binów (suma liczby trzęsień w binnie)
    return eq_m4["mag"].resample(f"{BIN_SIZE_MINUTES}min", origin=origin).sum()


# liczenie PPDF
def licz_ppdf_curve(eq_series, cr_series, windows_count, shift_bins, min_n=100):


    eq_arr = eq_series.values
    cr_arr = cr_series.values
    idx = eq_series.index
    n_bins = len(idx)

    wyniki = []

    # Przesuwanie okna po całej serii czasowej
    for s in range(0, n_bins - windows_count - abs(shift_bins)):
        # Określenie pozycji startowych dla EQ i CR z uwzględnieniem przesunięcia
        eq_start = s + shift_bins if shift_bins >= 0 else s
        cr_start = s if shift_bins >= 0 else s - shift_bins

        # Wycięcie okien
        cr_bins = cr_arr[cr_start: cr_start + windows_count]
        eq_bins = eq_arr[eq_start: eq_start + windows_count]

        # Sprawdzenie czy okna mają odpowiednią długość
        if len(cr_bins) < windows_count or len(eq_bins) < windows_count:
            continue

        # Obliczenie różnic CR między kolejnymi binami
        cr_delta = np.abs(np.diff(cr_bins))
        eq_sum = eq_bins[1:]          # suma trzęsień (pomijamy pierwszy bin)
        cr_mean_dopasowane = cr_bins[1:]  # wartości CR (pomijamy pierwszy bin)

        # Maska odrzucająca zera i braki danych
        maska = (cr_mean_dopasowane > 0) & (cr_delta > 0)
        eq_f = eq_sum[maska]
        cr_f = cr_delta[maska]

        n_total = len(eq_f)

        # Odrzucenie okien z zbyt małą liczbą obserwacji lub NaN
        if n_total < min_n or np.isnan(eq_f).any() or np.isnan(cr_f).any():
            continue

        # Normalizacja względem mediany
        med_eq = np.median(eq_f)
        med_cr = np.median(cr_f)
        if med_eq == 0 or med_cr == 0:
            continue

        A = (eq_f / med_eq) - 1
        B = (cr_f / med_cr) - 1
        C = np.sign(A * B)           # +1 jeśli A i B mają ten sam znak

        # Test dwumianowy: sprawdzenie czy proporcja +1 różni się od 0.5
        n_positive = int(np.sum(C == 1))
        p_val = binom.pmf(n_positive, n_total, 0.5)
        log10_p = np.log10(p_val) if p_val > 0 else np.nan

        t0 = idx[s]
        wyniki.append({"date": t0, "log10_PPDF": log10_p, "n_total": n_total, "n_positive": n_positive})

    return pd.DataFrame(wyniki)


# wykres
def rysuj_wykres(df_not_opt, df_opt):
    """Rysuje wykres porównawczy krzywych PPDF dla wariantów zoptymalizowanego i niezoptymalizowanego."""

    fig, ax = plt.subplots(figsize=(14, 7))

    # Krzywa zoptymalizowana (lag = 28600 min)
    ax.plot(df_opt["date"], df_opt["log10_PPDF"], color="tab:blue", linewidth=0.5,
             label=f"Optimized: lag = {LAG_OPTIMIZED_MIN} min")

    # Krzywa niezoptymalizowana (lag = 0)
    ax.plot(df_not_opt["date"], df_not_opt["log10_PPDF"], color="tab:orange", linewidth=0.5,
             label=f"Not optimized: lag = {LAG_NOT_OPTIMIZED_MIN}")

    ax.set_xlabel(r"$t_0$ (UTC)", fontsize=12)
    ax.set_ylabel(r"$\log_{10}(PPDF)$", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower left", fontsize=9)
    ax.set_title(
        f"The extreme PPDF (bin size = {BIN_SIZE_MINUTES} min.): optimized vs. not optimized\n"
        f"Optimized vs. not optimized: binomial PPDF ({STACJA})",
        fontsize=13,
    )
    fig.tight_layout()
    plt.savefig(WYJSCIE_WYKRES, dpi=300, bbox_inches="tight")
    plt.show()


# pipeline
def main():

    # Wczytanie danych
    cr_binned, origin = wczytaj_dane_cr_1h(SCIEZKA_CR)
    eq_binned = wczytaj_dane_eq(SCIEZKA_EQ, origin=origin)

    # Wspólny zakres czasowy = PRZECIĘCIE (nie unia) - okno nie może zahaczać o brakujące dane
    data_start = max(eq_binned.index.min(), cr_binned.index.min())
    data_end = min(eq_binned.index.max(), cr_binned.index.max())
    wspolny_indeks = pd.date_range(data_start, data_end, freq=f"{BIN_SIZE_MINUTES}min")

    # Reindeksowanie do wspólnej siatki czasowej
    eq_binned = eq_binned.reindex(wspolny_indeks, fill_value=0)
    cr_binned = cr_binned.reindex(wspolny_indeks)

    # Obliczenie krzywej niezoptymalizowanej (lag = 0)
    df_not_opt = licz_ppdf_curve(eq_binned, cr_binned, WINDOWS_COUNT, SHIFT_NOT_OPT_BINS, min_n=MIN_N)

    # Obliczenie krzywej zoptymalizowanej (lag = 28600 min)
    df_opt = licz_ppdf_curve(eq_binned, cr_binned, WINDOWS_COUNT, SHIFT_OPT_BINS, min_n=MIN_N)

    # Zapis wyników do plików CSV
    df_not_opt.to_csv(WYJSCIE_CSV.replace(".csv", "_not_optimized.csv"), index=False)
    df_opt.to_csv(WYJSCIE_CSV.replace(".csv", "_optimized.csv"), index=False)

    # Generowanie wykresu
    rysuj_wykres(df_not_opt, df_opt)


if __name__ == "__main__":
    main()